# Experiment: C13 Architecture Scout — BottleneckAttnUNet3D (MSE-only)

**Date:** 2026-03-02  
**Experiment ID:** `C13_bottleneck_mse` (seed 42, single seed)  
**Status:** Complete (Preliminary — seed 42 only)  
**Type:** Training (architecture scout)  
**GitHub Issue:** [#53](https://github.com/wrockey/vmat-diffusion/issues/53)  

---

## 1. Overview

### 1.1 Objective

Test whether BottleneckAttnUNet3D architecture improves dose prediction versus the baseline U-Net, using MSE-only loss to isolate the architecture effect. This is scout C13 in the architecture ablation series (#53), evaluating whether attention blocks at the bottleneck (deepest encoder/decoder levels) provide a meaningful benefit.

### 1.2 Hypothesis

Attention blocks at the bottleneck will allow the model to capture long-range spatial dependencies at the coarsest resolution, where receptive fields are largest. This should improve global dose distribution coherence and PTV coverage (D95 gap, PTV gamma) relative to baseline, which uses standard convolution blocks throughout.

### 1.3 Key Results

| Metric | C13 BottleneckAttn | Baseline (seed 42) | Delta |
|--------|-------------------|-------------------|-------|
| MAE (Gy) | 4.91 ± 2.87 | 4.80 ± 2.45 | +0.11 (slightly worse) |
| Gamma Global (%) | 27.7 ± 11.6 | 28.1 ± 12.6 | -0.4pp (slightly worse) |
| Gamma PTV (%) | 84.0 ± 11.2 | 87.3 ± 10.8 | **-3.3pp (worse)** |
| D95 Gap (Gy) | -1.47 ± 1.16 | -0.83 ± 0.46 | **-0.64 (worse)** |

### 1.4 Conclusion

**BottleneckAttnUNet3D does not improve over baseline.** All four primary metrics are slightly worse: MAE increases by 0.11 Gy, global gamma decreases by 0.4pp, PTV gamma drops by 3.3pp (84.0% vs 87.3%), and D95 underdose deepens by 0.64 Gy (-1.47 vs -0.83 Gy). The bottleneck attention mechanism does not help and mildly hurts PTV coverage. Comparing the two architecture scouts, C13 (bottleneck attention) is actually less bad than C11 (skip-connection attention gates): PTV gamma 84.0% vs 81.1%, D95 gap -1.47 vs -2.20 Gy. But both are worse than baseline. Architecture is clearly not the bottleneck — loss function engineering (as demonstrated by the combined loss pilot at 96.4% PTV gamma) has a far larger impact. Training was interrupted by a power outage and resumed with the `--resume` flag; total wall time was approximately 15.7h.

---

## 2. What Changed

Compared to baseline_v23 (seed 42), this experiment replaces **BaselineUNet3D with BottleneckAttnUNet3D**. **Everything else is identical** (same loss, data, augmentation, seed, epochs, optimizer, batch size).

| Parameter | Baseline seed42 | This Experiment |
|-----------|----------------|------------------|
| Architecture | BaselineUNet3D | **BottleneckAttnUNet3D** |
| Parameters | 23.73M | **24.32M (+2.5%)** |
| Bottleneck | Standard conv blocks | **Attention-augmented conv blocks** |
| Loss function | MSE + neg penalty | MSE + neg penalty (identical) |
| Seed | 42 | 42 (identical split) |
| Augmentation | ON | ON (identical) |
| Optimizer | AdamW, lr=1e-4, wd=0.01 | AdamW, lr=1e-4, wd=0.01 (identical) |
| Epochs | 200 | 200 (identical) |
| Batch size | 2 | 2 (identical) |
| Patch size | 128³ | 128³ (identical) |
| All other hyperparameters | Default | Default (identical) |

**Single variable under test:** BaselineUNet3D (standard conv blocks at bottleneck) vs BottleneckAttnUNet3D (attention-augmented conv blocks at bottleneck).

---

## 3. Reproducibility

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()

REPRODUCIBILITY = {
    'git_commit': '9eaeffa',
    'python_version': '3.12.12',
    'pytorch_version': '2.10.0+cu126',
    'pytorch_lightning_version': '2.6.1',
    'cuda_version': '12.6',
    'gpu': 'NVIDIA GeForce RTX 3090',
    'random_seed': 42,
    'experiment_date': '2026-03-02',
    'platform': 'WSL2 Ubuntu 24.04 LTS',
    'training_time_hours_resumed': 3.71,
    'training_time_hours_total': 15.7,
    'note': 'Training interrupted by power outage at ~epoch 148, resumed with --resume flag',
}

print('Reproducibility Information:')
for k, v in REPRODUCIBILITY.items():
    print(f'  {k}: {v}')

### Command to Reproduce

```bash
# Train (BottleneckAttnUNet3D, MSE-only loss)
python scripts/train_baseline_unet.py \
    --data_dir ~/data/processed_npz \
    --exp_name C13_bottleneck_mse_seed42 \
    --architecture bottleneck_attn \
    --epochs 200 --batch_size 2 --seed 42

# Resume after power outage (same command with --resume)
python scripts/train_baseline_unet.py \
    --data_dir ~/data/processed_npz \
    --exp_name C13_bottleneck_mse_seed42 \
    --architecture bottleneck_attn \
    --epochs 200 --batch_size 2 --seed 42 --resume

# Inference
python scripts/inference_baseline_unet.py \
    --checkpoint runs/C13_bottleneck_mse_seed42/checkpoints/best-epoch=146-val/mae_gy=6.416.ckpt \
    --input_dir ~/data/test_npz \
    --output_dir predictions/C13_bottleneck_mse_seed42_test \
    --compute_metrics --overlap 64 --gamma_subsample 4
```

Environment snapshot: `runs/C13_bottleneck_mse_seed42/environment_snapshot.txt`

---

## 4. Dataset

In [ ]:
import json
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()

test_cases_path = PROJECT_ROOT / 'runs' / 'C13_bottleneck_mse_seed42' / 'test_cases.json'
with open(test_cases_path) as f:
    test_info = json.load(f)

print(f'Preprocessing version: v2.3.0')
print(f'Total cases: 74')
print(f'Split (seed={test_info["seed"]}): 60 train / 7 val / 7 test')
print(f'Test case IDs: {sorted(test_info["test_cases"])}')
print(f'\nNote: Same seed/split as baseline_v23 seed42 and C11 for direct architecture comparison.')

**Test cases (7):** prostate70gy_0005, prostate70gy_0018, prostate70gy_0024, prostate70gy_0027, prostate70gy_0056, prostate70gy_0065, prostate70gy_0079

**Data provenance:** 74 cases preprocessed with v2.3.0 pipeline (native resolution crop, B-spline dose resampling). Identical to baseline_v23 and C11. The same seed 42 split ensures direct comparability — the only variable is the architecture.

---

## 5. Model & Training Configuration

In [ ]:
import json
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()

config_path = PROJECT_ROOT / 'runs' / 'C13_bottleneck_mse_seed42' / 'training_config.json'
with open(config_path) as f:
    config = json.load(f)

print(f'Model: {config["model"]}')
print(f'Parameters: {config["model_params"]:,}')

print(f'\nHyperparameters:')
for k, v in sorted(config['hparams'].items()):
    print(f'  {k}: {v}')

summary_path = PROJECT_ROOT / 'runs' / 'C13_bottleneck_mse_seed42' / 'training_summary.json'
with open(summary_path) as f:
    summary = json.load(f)

print(f'\nTraining Summary:')
print(f'  Duration (resumed portion): {summary["total_time_hours"]:.2f} hours')
print(f'  Best val MAE: {summary["best_val_mae_gy"]:.3f} Gy')
print(f'  Final epoch: {summary["final_metrics"]["epoch"]}')
print(f'  Note: Training interrupted by power outage, resumed with --resume flag')

### Architecture

- **Model:** BottleneckAttnUNet3D, 48 base channels (48 -> 96 -> 192 -> 384 -> 768), **24,323,777 parameters** (vs 23.73M baseline, +2.5%)
- **Input:** 9 channels (1 CT + 8 SDF), **Output:** 1 channel (dose)
- **Constraint conditioning:** FiLM embedding (13-dim constraint vector)
- **Patch size:** 128x128x128 voxels
- **Key difference:** Attention-augmented convolution blocks at the deepest encoder and decoder levels (bottleneck). Standard conv blocks are used at all other resolution levels — skip connections are unmodified (no attention gates, unlike C11).

### Bottleneck Attention Mechanism

Unlike C11 (AttentionUNet3D) which adds attention gates at every skip connection, BottleneckAttnUNet3D concentrates the attention mechanism at the bottleneck — the deepest encoder/decoder levels where the feature maps have the smallest spatial dimensions and the largest receptive fields. The standard conv blocks at the bottleneck are replaced with attention-augmented blocks that can model long-range spatial dependencies at the coarsest resolution.

This design trades off local attention (C11's skip-connection gates) for global attention (bottleneck self-attention). The hypothesis is that long-range context at the bottleneck may be more useful than selective feature gating at skip connections, since dose distributions have global spatial coherence (the dose at one point depends on the entire beam arrangement).

### Loss Configuration

| Component | Weight | Notes |
|-----------|--------|-------|
| MSE | 1.0 | Standard pixel-wise mean squared error |
| Negative penalty | 0.1 | Penalizes predicted dose < 0 |

MSE-only loss chosen to isolate the architecture effect from loss function effects. Identical to baseline_v23 seed42 and C11.

### Training

- **Optimizer:** AdamW, lr=1e-4, weight_decay=0.01
- **Epochs:** 200, batch_size=2
- **Best checkpoint:** epoch 146 (val MAE = 6.416 Gy)
- **Training time:** ~12h initial run (interrupted by power outage at ~epoch 148) + 3.71h resumed portion. Total wall time approximately 15.7h.
- **Resume:** Training was interrupted by a power outage. Resumed from last checkpoint with `--resume` flag. The best checkpoint (epoch 146) was saved before the outage.
- **Augmentation:** ON (random flips + intensity jitter)

---

## 6. Results

Figures generated by `scripts/generate_C13_bottleneck_mse_figures.py`.  
Representative case: **prostate70gy_0056** (below-median MAE = 3.15 Gy).  
Inference uses overlap=64, gamma_subsample=4.

### Per-Case Metrics

In [ ]:
import json
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
eval_path = PROJECT_ROOT / 'predictions' / 'C13_bottleneck_mse_seed42_test' / 'baseline_evaluation_results.json'

with open(eval_path) as f:
    results = json.load(f)

print(f'{"Case":<30} {"MAE (Gy)":>10} {"Gamma Gl (%)": >14} {"Gamma PTV (%)": >14} {"D95 Gap (Gy)": >13} {"PTV MAE (Gy)": >13}')
print('-' * 98)

maes, gammas_g, gammas_p, d95s, ptv_maes = [], [], [], [], []
for c in results['per_case_results']:
    cid = c['case_id']
    mae = c['dose_metrics']['mae_gy']
    gam_g = c['gamma']['global_3mm3pct']['gamma_pass_rate']
    gam_p = c['gamma']['ptv_region_3mm3pct']['gamma_pass_rate']
    d95 = c['dvh_metrics'].get('PTV70', {}).get('D95_error', float('nan'))
    ptv_mae = c['dose_metrics'].get('ptv_mae_gy', float('nan'))
    maes.append(mae)
    gammas_g.append(gam_g)
    gammas_p.append(gam_p)
    d95s.append(d95)
    ptv_maes.append(ptv_mae)
    print(f'{cid:<30} {mae:>10.2f} {gam_g:>14.1f} {gam_p:>14.1f} {d95:>13.2f} {ptv_mae:>13.2f}')

print('-' * 98)
print(f'{"Mean +/- Std":<30} '
      f'{np.mean(maes):>10.2f}+/-{np.std(maes):.2f} '
      f'{np.mean(gammas_g):>10.1f}+/-{np.std(gammas_g):.1f} '
      f'{np.mean(gammas_p):>10.1f}+/-{np.std(gammas_p):.1f} '
      f'{np.mean(d95s):>9.2f}+/-{np.std(d95s):.2f} '
      f'{np.mean(ptv_maes):>9.2f}+/-{np.std(ptv_maes):.2f}')

**Approximate per-case metrics (from evaluation JSON — load code above for exact values):**

| Case | MAE (Gy) | Gamma Global (%) | Gamma PTV (%) | D95 Gap (Gy) | PTV MAE (Gy) |
|------|----------|-----------------|---------------|-------------|-------------|
| prostate70gy_0005 | 5.31 | 20.7 | 97.5 | -0.36 | 0.78 |
| prostate70gy_0018 | 5.78 | 13.4 | 81.9 | -1.35 | 1.71 |
| prostate70gy_0024 | 5.67 | 11.4 | 82.2 | -0.05 | 2.11 |
| prostate70gy_0027 | 1.72 | 42.5 | 89.6 | -1.16 | 1.20 |
| prostate70gy_0056 | 3.15 | 37.2 | 88.6 | -2.17 | 1.21 |
| prostate70gy_0065 | 10.77 | 30.9 | 88.9 | -1.39 | 0.75 |
| prostate70gy_0079 | 1.96 | 37.5 | 59.1 | -3.84 | 2.75 |
| **Mean +/- Std** | **4.91 +/- 2.87** | **27.7 +/- 11.6** | **84.0 +/- 11.2** | **-1.47 +/- 1.16** | **1.50 +/- 0.68** |

**Notable:** prostate70gy_0005 achieves 97.5% PTV Gamma (the only case above the 95% clinical target), but this is not representative — the next best is 89.6%. All D95 gaps are negative (underdose), ranging from -0.05 to -3.84 Gy. prostate70gy_0065 is again the highest MAE outlier (10.77 Gy). prostate70gy_0079 shows the worst PTV Gamma (59.1%) and deepest D95 underdose (-3.84 Gy).

### 6.1 Training Curves

In [ ]:
from IPython.display import Image, display
display(Image(filename='../runs/C13_bottleneck_mse/figures/fig1_training_curves.png', width=900))

**Caption:** Training loss and validation MAE vs epoch for C13 BottleneckAttnUNet3D (seed 42, 200 epochs). Best val MAE: 6.416 Gy at epoch 146. Training was interrupted by a power outage at approximately epoch 148 and resumed with the `--resume` flag.

**Key observations:**
- Best val MAE (6.416 Gy) is worse than both baseline (6.05 Gy) and C11 AttentionUNet (6.40 Gy), suggesting the bottleneck attention mechanism does not improve generalization under MSE-only loss
- Training appears stable with no divergence before or after the resume, confirming the bottleneck attention implementation is numerically sound and resume handled correctly
- The extra 0.6M attention parameters (+2.5% over baseline) do not improve convergence
- **Clinical implication:** The bottleneck attention adds compute overhead without reducing validation loss. Like C11, the architecture modification alone cannot compensate for MSE-only training limitations.

### 6.2 Dose Colorwash

In [ ]:
from IPython.display import Image, display
display(Image(filename='../runs/C13_bottleneck_mse/figures/fig2_dose_colorwash.png', width=900))

**Caption:** Predicted vs ground truth dose for prostate70gy_0056 (MAE = 3.15 Gy, below-median). Axial, coronal, sagittal views through PTV70 centroid.

**Key observations:**
- PTV70 region shows mildly cooler dose (less red/orange) in prediction vs GT, consistent with the -2.17 Gy D95 underdose for this case
- Overall dose shape and conformality are preserved — the model correctly identifies the treatment region
- The underdose is less severe than C11 for this case (D95 gap -2.17 vs -2.53 Gy), consistent with the aggregate trend that C13 is less bad than C11
- **Clinical implication:** The bottleneck attention mechanism fails to improve PTV boundary accuracy. The predicted dose undershoots the prescription near PTV edges, similar to baseline and C11. The pattern is qualitatively similar to all MSE-only experiments — architecture does not change the fundamental dose distribution behavior.

### 6.3 Dose Difference Map

In [ ]:
from IPython.display import Image, display
display(Image(filename='../runs/C13_bottleneck_mse/figures/fig3_dose_difference.png', width=900))

**Caption:** Dose difference (predicted minus GT, Gy) for prostate70gy_0056. Blue = underdose, red = overdose.

**Key observations:**
- PTV region shows predominantly blue (underdose), consistent with baseline and C11 behavior under MSE-only training
- The spatial pattern of underdose at PTV boundaries is similar across all three architectures (baseline, C11, C13), confirming it is a loss-driven artifact
- Peripheral low-dose regions show mixed blue/red consistent with baseline behavior
- **Clinical implication:** The bottleneck attention mechanism does not change the spatial error pattern. The underdose at PTV edges is characteristic of MSE optimization without explicit PTV-focused loss terms. Combined with C11 results, this is now the second architecture scout confirming that loss engineering — not architecture — is the key lever for PTV coverage.

### 6.4 DVH Comparison

In [ ]:
from IPython.display import Image, display
display(Image(filename='../runs/C13_bottleneck_mse/figures/fig4_dvh_comparison.png', width=800))

**Caption:** DVH curves for prostate70gy_0056. Solid = ground truth, dashed = predicted.

**Key observations:**
- PTV70 predicted DVH is shifted left (lower dose) compared to GT, consistent with the -2.17 Gy D95 underdose
- OAR DVH curves track GT at a similar level to baseline and C11 — the bottleneck attention mechanism does not degrade OAR accuracy
- The PTV56 DVH is also shifted left, indicating the underdose extends to the lower-dose PTV
- The DVH pattern is qualitatively indistinguishable from baseline and C11 under MSE-only training
- **Clinical implication:** A clinical plan with this predicted DVH would fail PTV coverage requirements. The D95 underdose would prompt a treatment plan revision. This is the same failure mode as baseline and C11, confirming architecture does not address the MSE-driven underdose bias.

### 6.5 Gamma Analysis

In [ ]:
from IPython.display import Image, display
display(Image(filename='../runs/C13_bottleneck_mse/figures/fig5_gamma_bar_chart.png', width=900))

**Caption:** Global vs PTV-region Gamma 3%/3mm per test case (C13 BottleneckAttn MSE, seed 42).

**Key observations:**
- One case (prostate70gy_0005) exceeds the 95% PTV Gamma clinical target at 97.5%, but this is an outlier — no other case exceeds 90%
- Mean PTV Gamma of 84.0% is below the 95% target and below baseline (87.3%), but better than C11 (81.1%)
- Global Gamma (27.7%) is comparable to baseline (28.1%) and C11 (29.6%)
- prostate70gy_0079 shows the worst PTV Gamma at 59.1%, substantially worse than all other cases
- **Clinical implication:** BottleneckAttnUNet3D fails to meet clinical spatial accuracy standards for PTV coverage. The 3.3pp regression vs baseline is smaller than C11's 6.1pp regression, suggesting bottleneck attention is less harmful than skip-connection attention, but still does not help. The 95% clinical target requires loss engineering, not architecture changes.

### 6.6 Per-Case Box Plots

In [ ]:
from IPython.display import Image, display
display(Image(filename='../runs/C13_bottleneck_mse/figures/fig6_per_case_boxplots.png', width=900))

**Caption:** Metric distributions across 7 test cases (C13 BottleneckAttn MSE, seed 42).

**Key observations:**
- D95 gap distribution is entirely negative (all 7 cases underdose), ranging from -0.05 to -3.84 Gy
- MAE distribution has high variance (2.87 Gy std), driven by the prostate70gy_0065 outlier (10.77 Gy)
- PTV Gamma distribution spans 59.1% to 97.5% — very wide spread (11.2% std), with one strong outlier above 95% but the rest below 90%
- prostate70gy_0027 is again the easiest case (1.72 Gy MAE) and prostate70gy_0065 the hardest (10.77 Gy MAE) — anatomy/plan complexity drives case difficulty more than architecture
- **Clinical implication:** The wide variance and systematic underdose pattern across all 7 cases indicates this is a bias, not noise. The bottleneck attention architecture does not correct the fundamental underdose tendency of MSE-only training. The per-case ranking is nearly identical across baseline, C11, and C13, confirming case difficulty is data-driven.

### 6.7 QUANTEC Compliance

In [ ]:
from IPython.display import Image, display
display(Image(filename='../runs/C13_bottleneck_mse/figures/fig7_quantec_compliance.png', width=900))

**Caption:** QUANTEC constraint compliance heatmap (C13 BottleneckAttn MSE, seed 42).

**Key observations:**
- Volume-based OAR constraints pass universally, consistent with baseline and C11 — the model correctly preserves OAR sparing regardless of architecture
- PTV D95 constraints fail (predicted dose is below threshold) due to underdose, similar to baseline and C11
- The compliance pattern is nearly identical across all three architectures — further evidence that compliance behavior is loss-driven, not architecture-driven
- **Clinical implication:** BottleneckAttnUNet3D meets OAR constraints but fails PTV coverage. This is the same failure mode as baseline and C11, confirming the bottleneck attention blocks do not address the root cause (lack of PTV-targeted loss).

### 6.8 Femur Asymmetry

In [ ]:
from IPython.display import Image, display
display(Image(filename='../runs/C13_bottleneck_mse/figures/fig8_femur_asymmetry.png', width=900))

**Caption:** Femur asymmetry analysis (C13 BottleneckAttn MSE, seed 42). Note: this is a single-seed pilot (seed 42 only). The figure shows per-case femur dose asymmetry rather than cross-seed comparisons.

**Key observations:**
- Single-seed pilot — cross-seed variability cannot be assessed
- Femur left/right asymmetry patterns are consistent with baseline and C11
- The bottleneck attention does not appear to improve bilateral symmetry in the predicted dose
- **Clinical implication:** Given the negative result (-3.3pp PTV gamma vs baseline), full 3-seed confirmation is not prioritized. The effect size is clear enough from a single seed that the direction of the finding is established: bottleneck attention does not help.

---

## 7. Statistical Analysis

This is a **single-seed pilot** (seed 42 only). Formal cross-seed statistics are not available. The comparison below is a **paired analysis** on the same 7 test cases (same seed, same split) between this experiment, the baseline, and C11.

In [ ]:
import json
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
pred_base = PROJECT_ROOT / 'predictions'

def load_metrics(eval_path):
    with open(eval_path) as f:
        d = json.load(f)
    maes, gammas_g, gammas_p, d95 = [], [], [], []
    for c in d['per_case_results']:
        maes.append(c['dose_metrics']['mae_gy'])
        gammas_g.append(c['gamma']['global_3mm3pct']['gamma_pass_rate'])
        gammas_p.append(c['gamma']['ptv_region_3mm3pct']['gamma_pass_rate'])
        ptv70 = c['dvh_metrics'].get('PTV70', {})
        if 'D95_error' in ptv70:
            d95.append(ptv70['D95_error'])
    return {'mae': maes, 'gamma_g': gammas_g, 'gamma_p': gammas_p, 'd95': d95,
            'case_ids': [c['case_id'] for c in d['per_case_results']]}

c13 = load_metrics(pred_base / 'C13_bottleneck_mse_seed42_test/baseline_evaluation_results.json')
c11 = load_metrics(pred_base / 'C11_attn_mse_seed42_test/baseline_evaluation_results.json')
baseline = load_metrics(pred_base / 'baseline_v23_seed42_test/baseline_evaluation_results.json')

print('Head-to-Head Comparison: C13 BottleneckAttn vs Baseline vs C11 (same 7 test cases, same seed 42 split)')
print('=' * 100)
for metric, key, unit in [('MAE', 'mae', 'Gy'), ('Gamma Global', 'gamma_g', '%'),
                            ('Gamma PTV', 'gamma_p', '%'), ('D95 Gap', 'd95', 'Gy')]:
    c13_m, c13_s = np.mean(c13[key]), np.std(c13[key])
    c11_m, c11_s = np.mean(c11[key]), np.std(c11[key])
    bl_m, bl_s = np.mean(baseline[key]), np.std(baseline[key])
    diff_bl = c13_m - bl_m
    diff_c11 = c13_m - c11_m
    sign_bl = '+' if diff_bl > 0 else ''
    sign_c11 = '+' if diff_c11 > 0 else ''
    print(f'  {metric:<18} C13: {c13_m:6.2f}+/-{c13_s:5.2f} {unit}  '
          f'Baseline: {bl_m:6.2f}+/-{bl_s:5.2f} {unit}  '
          f'C11: {c11_m:6.2f}+/-{c11_s:5.2f} {unit}  '
          f'vs BL: {sign_bl}{diff_bl:.2f}  vs C11: {sign_c11}{diff_c11:.2f}')

# Per-case paired differences for PTV Gamma (C13 vs Baseline)
print(f'\nPer-Case PTV Gamma Differences (C13 - Baseline):')
diffs_gamma_bl = []
for i, cid in enumerate(c13['case_ids']):
    j = baseline['case_ids'].index(cid)
    d = c13['gamma_p'][i] - baseline['gamma_p'][j]
    diffs_gamma_bl.append(d)
    sign = '+' if d > 0 else ''
    print(f'  {cid}: {sign}{d:.1f}pp')
print(f'  Mean diff: {np.mean(diffs_gamma_bl):+.1f}pp (positive = C13 is better)')
print(f'  Cases where C13 is better: {sum(1 for d in diffs_gamma_bl if d > 0)}/7')

# Per-case paired differences for PTV Gamma (C13 vs C11)
print(f'\nPer-Case PTV Gamma Differences (C13 - C11):')
diffs_gamma_c11 = []
for i, cid in enumerate(c13['case_ids']):
    j = c11['case_ids'].index(cid)
    d = c13['gamma_p'][i] - c11['gamma_p'][j]
    diffs_gamma_c11.append(d)
    sign = '+' if d > 0 else ''
    print(f'  {cid}: {sign}{d:.1f}pp')
print(f'  Mean diff: {np.mean(diffs_gamma_c11):+.1f}pp (positive = C13 is better than C11)')
print(f'  Cases where C13 is better than C11: {sum(1 for d in diffs_gamma_c11 if d > 0)}/7')

# Per-case paired differences for D95 (C13 vs Baseline)
print(f'\nPer-Case D95 Gap Differences (C13 - Baseline):')
diffs_d95 = []
for i, cid in enumerate(c13['case_ids']):
    j = baseline['case_ids'].index(cid)
    d = c13['d95'][i] - baseline['d95'][j]
    diffs_d95.append(d)
    sign = '+' if d > 0 else ''
    print(f'  {cid}: {sign}{d:.2f} Gy')
print(f'  Mean diff: {np.mean(diffs_d95):+.2f} Gy (negative = C13 underdoses more)')
print(f'  Cases where C13 is better (less underdose): {sum(1 for d in diffs_d95 if d > 0)}/7')

print(f'\nNote: Pilot status (1 seed). Architecture scout is negative result -- full 3-seed '
      f'confirmation not planned.')

### Statistical Summary

| Metric | C13 BottleneckAttn | Baseline (seed42) | C11 AttentionUNet | Delta (C13 vs BL) | Delta (C13 vs C11) |
|--------|--------------------|-------------------|-------------------|--------------------|---------------------|
| MAE (Gy) | 4.91 +/- 2.87 | 4.80 +/- 2.45 | 4.57 +/- 2.51 | +0.11 (worse) | +0.34 (worse) |
| Gamma global (%) | 27.7 +/- 11.6 | 28.1 +/- 12.6 | 29.6 +/- 9.5 | -0.4pp (worse) | -1.9pp (worse) |
| Gamma PTV (%) | 84.0 +/- 11.2 | 87.3 +/- 10.8 | 81.1 +/- 8.8 | **-3.3pp (worse)** | **+2.9pp (better)** |
| D95 gap (Gy) | -1.47 +/- 1.16 | -0.83 +/- 0.46 | -2.20 +/- 0.91 | **-0.64 (worse)** | **+0.73 (better)** |

**Interpretation:** The C13 architecture scout is a **negative result** vs baseline. All four primary metrics are slightly worse. However, C13 is less bad than C11 on the two primary clinical metrics: PTV gamma is 84.0% vs 81.1% (+2.9pp) and D95 gap is -1.47 vs -2.20 Gy (+0.73 Gy). This suggests that bottleneck attention is less harmful than skip-connection attention, but neither architecture helps compared to baseline.

The relative ranking of the three architectures on PTV gamma is: **Baseline (87.3%) > C13 Bottleneck (84.0%) > C11 Attention (81.1%)**. On D95 gap: **Baseline (-0.83) > C13 (-1.47) > C11 (-2.20)**. The plain baseline with standard skip connections remains the best architecture under MSE-only training.

This is a single-seed pilot. Given the magnitude and consistency of the negative effect, full 3-seed confirmation is not warranted. The architecture scout series is converging on the conclusion that **architecture choice is not the primary bottleneck** for clinical metric improvement.

---

## 8. Cross-Experiment Comparison

| Experiment | MAE (Gy) | Gamma Global (%) | Gamma PTV (%) | D95 Gap (Gy) | Status |
|------------|----------|-----------------|---------------|-------------|--------|
| Baseline 3-seed aggregate | 4.22 +/- 0.53 | 33.8 +/- 4.6 | 80.2 +/- 5.3 | -1.76 +/- 0.69 | Complete |
| Baseline seed42 | 4.80 +/- 2.45 | 28.1 +/- 12.6 | 87.3 +/- 10.8 | -0.83 +/- 0.46 | Complete |
| No augmentation (seed42) | 5.04 +/- 2.92 | 27.4 +/- 9.8 | 83.2 +/- 9.8 | -1.89 +/- 1.01 | Complete |
| Combined loss pilot (seed42) | 4.54 +/- 1.84 | 30.8 +/- 12.4 | **96.4 +/- 5.4** | +1.37 +/- 0.57 | Preliminary |
| C11 AttentionUNet MSE (seed42) | 4.57 +/- 2.51 | 29.6 +/- 9.5 | 81.1 +/- 8.8 | -2.20 +/- 0.91 | Preliminary |
| **C13 BottleneckAttn MSE (seed42)** | **4.91 +/- 2.87** | **27.7 +/- 11.6** | **84.0 +/- 11.2** | **-1.47 +/- 1.16** | **Preliminary** |
| Phase 2 target | < 3.0 | -- | > 95% | >= -0.5 | -- |

### Key Takeaways

1. **Architecture is not the bottleneck — confirmed by two scouts.** Both C11 (AttentionUNet, -6.1pp PTV gamma) and C13 (BottleneckAttn, -3.3pp PTV gamma) perform worse than the baseline plain U-Net. Architecture changes without loss changes cannot achieve the 95% PTV Gamma target.

2. **Loss function engineering dominates.** The combined loss pilot achieves 96.4% PTV Gamma with the *same* baseline architecture — that is +12.4pp over C13 and +15.3pp over C11. The difference is entirely attributable to the loss function, not architecture. This makes it clear that the next productive investment is loss tuning, not more architecture experiments.

3. **Bottleneck attention is less harmful than skip-connection attention.** C13 (84.0% PTV gamma, -1.47 Gy D95) beats C11 (81.1%, -2.20 Gy) on both clinical metrics. Skip-connection attention gates (C11) actively suppress important encoder features; bottleneck attention (C13) at least leaves skip connections intact. But neither helps.

4. **MSE-only training systematically underdoses.** All MSE-only experiments (baseline, no-aug, C11, C13) show negative D95 gaps. The combined loss is the only experiment to flip the sign (overdose), confirming that explicit PTV-targeted loss terms are required for PTV coverage.

5. **Case difficulty is data-driven, not architecture-driven.** The per-case ranking (prostate70gy_0027 easy, prostate70gy_0065 hard) is remarkably consistent across all three architectures, suggesting that patient anatomy and plan complexity dominate case-level performance, not model capacity.

6. **Architecture scouts should be concluded.** With two negative scouts (C11, C13), the evidence is strong that moderate architecture changes do not help under MSE-only training. The remaining C15 (wider baseline) scout may be worth running as a final data point, but the strategic conclusion is already clear: invest in loss engineering.

---

## 9. Conclusions, Limitations, and Next Steps

### What Worked

1. **Stable training despite power outage.** BottleneckAttnUNet3D trains without divergence and successfully resumes from checkpoint after interruption. The `--resume` flag correctly restores training state.

2. **OAR sparing maintained.** QUANTEC OAR compliance is equivalent to baseline and C11. The bottleneck attention blocks do not degrade OAR dose accuracy.

3. **Less harmful than C11.** PTV gamma (84.0% vs 81.1%) and D95 gap (-1.47 vs -2.20 Gy) are both closer to baseline with bottleneck attention than with skip-connection attention. This provides a mechanistic insight: disrupting skip connections (C11) is more harmful than modifying the bottleneck (C13).

4. **Architecture scout validated the experimental framework.** The C11/C13 scout series is successfully isolating architecture from loss effects, enabling clear attribution. Both scouts point to the same conclusion.

### What Didn't Work

1. **PTV gamma regressed by 3.3pp** (84.0% vs 87.3% for baseline). The bottleneck attention mechanism does not improve PTV coverage under MSE-only training.

2. **D95 underdose deepened by 0.64 Gy** (-1.47 vs -0.83 Gy). All 7 test cases show underdose (negative D95 gap), though one case (prostate70gy_0024 at -0.05 Gy) is nearly zero.

3. **MAE slightly worse** (4.91 vs 4.80 Gy). Unlike C11 which marginally improved MAE, C13 is slightly worse on all metrics.

4. **Best val MAE worse** (6.416 Gy vs 6.05 Gy for baseline). The extra parameters do not improve validation loss.

### Mechanistic Hypothesis

The bottleneck attention mechanism adds attention-augmented blocks at the deepest encoder/decoder levels, where feature maps are smallest (e.g., 4x4x4 at 768 channels). At this resolution, the receptive field already covers the entire input patch via stacked convolutions. Additional attention at the bottleneck may be redundant — the standard convolution blocks can already capture the global context that attention provides. This explains why C13 shows minimal change from baseline.

In contrast, C11's attention gates at skip connections actively interfere with the direct encoder-to-decoder information pathway that is critical for preserving spatial detail. This explains why C11 performs worse than C13 — skip connections are where the architecture adds value, and attention gates there can suppress useful features.

**Counter-hypothesis:** With a PTV-focused loss (e.g., the combined loss), bottleneck attention might help by learning to route dose-relevant global context more effectively. However, the minimal effect under MSE-only training suggests the bottleneck is not the architecture's limiting factor.

### Limitations

- **Single seed (42 only)** — results should be interpreted as directional evidence, not statistically confirmed. However, the finding is consistent with C11's negative result.
- **Small test set (n=7)** — cannot compute significance statistics. One outlier case can shift the mean substantially.
- **Power outage interrupted training** — the model was resumed with `--resume`. While the best checkpoint (epoch 146) was saved before the outage, the resumed training may have explored a slightly different trajectory. The effect on final metrics is likely negligible since the best checkpoint predates the outage.
- **Single attention variant** — only bottleneck attention at deepest levels tested. Attention at intermediate levels, or hybrid approaches, are not tested.

### Next Steps

1. **Conclude architecture scouts:** The C15 (wider baseline) scout is the remaining planned experiment in the architecture series. It tests whether more capacity (wider channels) helps without changing the architecture topology. Given C11 and C13 results, C15 is expected to show minimal improvement, but completing the series maintains scientific rigor and provides a complete comparison.

2. **Focus Phase 2 on loss engineering:** With two architecture scouts confirming that architecture is not the bottleneck, the highest-priority work is tuning the combined loss asymmetric penalty weight (from 3:1 to ~2:1) to correct the D95 overdose observed in the combined loss pilot.

3. **Do not run 3-seed BottleneckAttn confirmation:** The negative result is clear from seed 42 alone. Running 3 seeds on a clearly inferior architecture would consume GPU time without scientific value.

4. **Document architecture scout summary:** When all three scouts (C11, C13, C15) are complete, a summary analysis comparing all architecture variants should be added to the experiment index and potentially referenced in the publication methods section as evidence supporting the choice of the baseline architecture.

---

## 10. Artifacts

| Artifact | Path |
|----------|------|
| Run directory | `runs/C13_bottleneck_mse_seed42/` |
| Best checkpoint | `runs/C13_bottleneck_mse_seed42/checkpoints/best-epoch=146-val/mae_gy=6.416.ckpt` |
| Training config | `runs/C13_bottleneck_mse_seed42/training_config.json` |
| Training summary | `runs/C13_bottleneck_mse_seed42/training_summary.json` |
| Test cases | `runs/C13_bottleneck_mse_seed42/test_cases.json` |
| Predictions | `predictions/C13_bottleneck_mse_seed42_test/` |
| Evaluation JSON | `predictions/C13_bottleneck_mse_seed42_test/baseline_evaluation_results.json` |
| Figures (PNG + PDF) | `runs/C13_bottleneck_mse/figures/` (8 figures, 16 files) |
| Figure script | `scripts/generate_C13_bottleneck_mse_figures.py` |
| Environment snapshot | `runs/C13_bottleneck_mse_seed42/environment_snapshot.txt` |
| This notebook | `notebooks/2026-03-02_C13_bottleneck_mse.ipynb` |

---

*Notebook created: 2026-03-02*  
*Status: Complete (Preliminary — seed 42 only)*